# SI4006 · Sesión 10 · Extra: optimizar prompts con DSPy

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 3 · RAG

Este notebook acompaña la slide de DSPy. La idea es la que vimos al final de la sesión: hasta ahora
ajustamos los prompts a mano, pero una vez que tenemos una métrica (nuestro harness o RAGAS), podemos
dejar que una herramienta busque un mejor prompt de forma automática, midiendo cada intento contra esa
métrica. DSPy hace justo eso: describimos la tarea y damos unos ejemplos, y DSPy arma el prompt
(instrucciones y ejemplos) que maximiza el puntaje.

> Es un **extra**, no lo cubrimos a fondo en el curso. Corre con un modelo local vía Ollama, sin API
> keys. Con GPU (T4) va cómodo; en CPU funciona, pero lento.

## 0 · Setup: DSPy y un modelo local (Ollama)

Instalamos DSPy y levantamos un modelo pequeño con Ollama (`qwen2.5:1.5b`, el mismo tamaño que usamos
en el curso). Ollama expone un servidor compatible con OpenAI, que es como DSPy habla con el modelo.

In [ ]:
%pip install -q dspy
# Instala Ollama y descarga un modelo pequeño. La primera vez tarda (descarga ~1 GB).
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, os
subprocess.Popen(['ollama', 'serve'])   # servidor en segundo plano
time.sleep(5)
!ollama pull qwen2.5:1.5b

In [ ]:
import dspy
# DSPy habla con Ollama por su API compatible con OpenAI.
lm = dspy.LM('ollama_chat/qwen2.5:1.5b', api_base='http://localhost:11434', api_key='', temperature=0.0)
dspy.configure(lm=lm)
print('LM configurado:', lm.model)
# Alternativa con API key (si la tienen): dspy.LM('openai/gpt-4o-mini', api_key='sk-...')

## 1 · La tarea: clasificar el sentimiento de una reseña

Usamos una tarea simple para ver el efecto con claridad: decir si una reseña es positiva o negativa.
En DSPy no escribimos el prompt; describimos la tarea con una `Signature` (qué entra y qué sale).

In [ ]:
class Sentimiento(dspy.Signature):
    """Clasifica el sentimiento de una reseña de producto como 'positivo' o 'negativo'."""
    reseña: str = dspy.InputField()
    sentimiento: str = dspy.OutputField(desc="una sola palabra: positivo o negativo")

clasificar = dspy.Predict(Sentimiento)
print(clasificar(reseña='La batería dura nada y llegó rayado').sentimiento)

## 2 · Datos y métrica

Damos unos ejemplos etiquetados (train para que DSPy aprenda a armar el prompt, y dev para medir) y
una métrica: aquí, acertar la etiqueta. Esta métrica hace el papel que en el sistema real harían el
harness o RAGAS.

In [ ]:
def ej(r, s): return dspy.Example(reseña=r, sentimiento=s).with_inputs('reseña')
trainset = [
    ej('Me encanta, superó mis expectativas', 'positivo'),
    ej('Una compra excelente, lo recomiendo', 'positivo'),
    ej('Funciona de maravilla y llegó rápido', 'positivo'),
    ej('Pésimo, se dañó a la semana', 'negativo'),
    ej('No sirve para nada, perdí mi dinero', 'negativo'),
    ej('Llegó roto y el soporte no responde', 'negativo'),
]
devset = [
    ej('Cumple justo lo que promete, contento', 'positivo'),
    ej('Decepcionante, esperaba mucho más', 'negativo'),
    ej('Buen precio y buena calidad', 'positivo'),
    ej('Se calienta y se apaga solo', 'negativo'),
]

def acierta(ejemplo, pred, trace=None):
    return ejemplo.sentimiento.strip().lower() == pred.sentimiento.strip().lower()

## 3 · Baseline: el prompt sin optimizar

Medimos el clasificador tal cual, sin ejemplos en el prompt, sobre el dev set.

In [ ]:
from dspy.evaluate import Evaluate
evaluar = Evaluate(devset=devset, metric=acierta, num_threads=1, display_progress=True)
puntaje_base = evaluar(clasificar)
print('Puntaje baseline:', puntaje_base)

## 4 · Optimizar el prompt con DSPy

`BootstrapFewShot` prueba combinaciones de ejemplos del train, mide cada intento con nuestra métrica y
se queda con los que mejor puntúan. El resultado es un prompt con buenos ejemplos, elegido por datos y
no a mano.

In [ ]:
from dspy.teleprompt import BootstrapFewShot
optimizador = BootstrapFewShot(metric=acierta, max_bootstrapped_demos=3, max_labeled_demos=3)
clasificar_opt = optimizador.compile(clasificar, trainset=trainset)
puntaje_opt = evaluar(clasificar_opt)
print('Puntaje baseline :', puntaje_base)
print('Puntaje optimizado:', puntaje_opt)

## 5 · Ver qué prompt armó DSPy

El optimizado ya lleva ejemplos escogidos. Podemos ver el último prompt que se envió al modelo.

In [ ]:
print('Ejemplos que DSPy metió en el prompt:', len(clasificar_opt.demos))
clasificar_opt(reseña='La cámara es borrosa y la app se cierra')
dspy.inspect_history(n=1)   # muestra el prompt real que se mandó al modelo

## Lo que se llevan

El puntaje suele subir porque DSPy eligió, con datos, qué ejemplos e instrucciones meter en el prompt,
en vez de que los adivináramos nosotros. El punto de fondo: **sin una métrica confiable, optimizar el
prompt no tiene sentido; con ella, se puede automatizar.** Por eso este tema va después de aprender a
evaluar. Para su proyecto, la métrica sería su harness o RAGAS, y la tarea, su propio pipeline.